# 라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [5]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.mart_user_activation`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

c:\workspace\final_project\sns_service_analysis\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_id,signup_at,questionset_count,first_questionset_at,last_questionset_at,vote_count,first_vote_at,last_vote_at,selected_count,read_selected_count,first_selected_at,payment_count,first_payment_at,last_payment_at
0,868806,2023-05-02 03:30:58.454799+00:00,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT
1,868974,2023-05-02 04:18:16.334717+00:00,<NA>,NaT,NaT,<NA>,NaT,NaT,1,1,2023-05-07 06:25:02+00:00,<NA>,NaT,NaT
2,870723,2023-05-02 11:19:58.393218+00:00,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT
3,873327,2023-05-02 23:54:54.366413+00:00,<NA>,NaT,NaT,<NA>,NaT,NaT,52,52,2023-05-03 17:02:47+00:00,<NA>,NaT,NaT
4,880775,2023-05-04 12:21:00.258456+00:00,<NA>,NaT,NaT,<NA>,NaT,NaT,<NA>,<NA>,NaT,<NA>,NaT,NaT


## 데이터 확인

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   user_id               677080 non-null  Int64              
 1   signup_at             677080 non-null  datetime64[us, UTC]
 2   questionset_count     4972 non-null    Int64              
 3   first_questionset_at  4972 non-null    datetime64[us, UTC]
 4   last_questionset_at   4972 non-null    datetime64[us, UTC]
 5   vote_count            4849 non-null    Int64              
 6   first_vote_at         4849 non-null    datetime64[us, UTC]
 7   last_vote_at          4849 non-null    datetime64[us, UTC]
 8   selected_count        15426 non-null   Int64              
 9   read_selected_count   15426 non-null   Int64              
 10  first_selected_at     15426 non-null   datetime64[us, UTC]
 11  payment_count         59192 non-null   Int64              
 12 

## 결측 처리
- 날짜 데이터를 제외한 결측은 0으로 대체

In [7]:
count_cols = [
    'questionset_count',
    'vote_count',
    'selected_count',
    'read_selected_count',
    'payment_count'
]

df[count_cols] = df[count_cols].fillna(0)

In [ ]:
# 결측 대체 적용 체크

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   user_id               677080 non-null  Int64              
 1   signup_at             677080 non-null  datetime64[us, UTC]
 2   questionset_count     677080 non-null  Int64              
 3   first_questionset_at  4972 non-null    datetime64[us, UTC]
 4   last_questionset_at   4972 non-null    datetime64[us, UTC]
 5   vote_count            677080 non-null  Int64              
 6   first_vote_at         4849 non-null    datetime64[us, UTC]
 7   last_vote_at          4849 non-null    datetime64[us, UTC]
 8   selected_count        677080 non-null  Int64              
 9   read_selected_count   677080 non-null  Int64              
 10  first_selected_at     15426 non-null   datetime64[us, UTC]
 11  payment_count         677080 non-null  Int64              
 12 

In [12]:
df[df['signup_at'] < '2023-04-28']['vote_count'].value_counts()

vote_count
0       8591
273        1
41         1
166        1
256        1
567        1
250        1
168        1
22         1
170        1
88         1
553        1
246        1
217        1
2786       1
141        1
378        1
338        1
10         1
244        1
131        1
212        1
Name: count, dtype: Int64

In [13]:
pd.crosstab(
    df['questionset_count'] > 0,
    df['vote_count'] > 0,
    margins=True
)

vote_count,False,True,All
questionset_count,,,
False,672108,0,672108
True,123,4849,4972
All,672231,4849,677080


# 질문 및 투표참여 비율 파악

- 투표기록에 기록이 없고, 질문세트에도 기록이 없는 유저가 672,108명
- 질문 세트 없이 투표한 유저는 없음.
- 질문 세트는 열었지만 투표 기록에는 없는 유저가 123명
- 질문 세트를 열고 투표기록까지 남긴 유저가 4,849명

해당 내용을 해석해보자면 총 677,080명 중 투표가 기록된 유저 4,849명 약 0.7% 정도로 본다면 실제 투표 참여율은 가히 충격적인 수치..
또한, 투표한 모든 유저가 반드시 questionset 경험 기록을 가지고 있다고 보이며, 질문세트를 열고 투표한 기록이 투표기록 테이블에 적재가 되는 순서로 진행되는 것으로 보이며,
질문 세트를 열었지만 투표는 하지 않은 유저가 123명으로 투표 과정 중 이탈이라고 볼 수 있다.

반대로 해석해보자면 질문을 참여하기까지의 프로세스가 원활하지 않거나, 유저 개인의 성향으로 질문 투표를 하지 않는 등의 사유로 콘텐츠 이용이 원활하지않다고 볼 수 있다.
따라서, 해당 해석 결과를 토대로 콘텐츠 경험 이후 병목 보다, 콘텐츠 이용이 진행되지 않은 원인을 분석하는 것이 무엇보다 중요하다고 판단된다.


In [19]:
print("<포인트 구매 유저의 특성 확인>")

print(f"포인트 구매 경험이 있는 유저 : {df[(df['payment_count'] > 0)]['user_id'].nunique():,}")
print(f"질문 투표를 해보지 않고 포인트 구매한 유저 : {df[(df['payment_count'] > 0) & (df['vote_count'] == 0) & (df['questionset_count'] == 0)]['user_id'].nunique():,}")

print(f"투표 선택된 경험이 있는 유저 : {df[(df['selected_count'] > 0)]['user_id'].nunique():,}")
print(f"투표 선택된 경험이 있으면서 포인트를 구매한 유저 : {df[(df['selected_count'] > 0) & (df['payment_count'] > 0)]['user_id'].nunique():,}")

<포인트 구매 유저의 특성 확인>
포인트 구매 경험이 있는 유저 : 59,192
질문 투표를 해보지 않고 포인트 구매한 유저 : 58,782
투표 선택된 경험이 있는 유저 : 15,426
투표 선택된 경험이 있으면서 포인트를 구매한 유저 : 1,213


# 포인트를 구매한 유저 특성 파악
- 질문 투표를 경험한 유저가 약 4,800여명 정도이나, 포인트를 구매한 경험이 있는 유저가 59,192명

핵심 콘텐츠를 이용하지 않으면서 포인트를 결제한 유저가 다수 존재하며, 포인트 사용처가 대부분 투표를 보낸사람의 초성을 확인하거나, 채팅방을 여는 것에 사용하는 것으로 확인할 수 있었기 때문에 선행 조건인 투표를 받은 경험이 있는 유저가 포인트를 구매하는지 확인하기 위해 '투표를 받아본 경험이 있으면서, 포인트를 구매한 경험이 있는 유저'를 조건으로 확인한 결과 1,213명으로 예상과는 다르게 다른 이유로 결제한 유저가 더 많다는 것을 간접적으로 확인이 가능했다.

따라서, 투표에 선택된 경험이 포인트 구매에 미치는 영향이 크지 않다고 볼 수 있다.

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677080 entries, 0 to 677079
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   user_id               677080 non-null  Int64              
 1   signup_at             677080 non-null  datetime64[us, UTC]
 2   questionset_count     677080 non-null  Int64              
 3   first_questionset_at  4972 non-null    datetime64[us, UTC]
 4   last_questionset_at   4972 non-null    datetime64[us, UTC]
 5   vote_count            677080 non-null  Int64              
 6   first_vote_at         4849 non-null    datetime64[us, UTC]
 7   last_vote_at          4849 non-null    datetime64[us, UTC]
 8   selected_count        677080 non-null  Int64              
 9   read_selected_count   677080 non-null  Int64              
 10  first_selected_at     15426 non-null   datetime64[us, UTC]
 11  payment_count         677080 non-null  Int64              
 12 